### Embedding and Vector Store DB

In [11]:

import os
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [8]:
### Read all the pdf's inside the directory
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)
    
    # Find all PDF files recursively
    pdf_files = list(pdf_dir.glob("**/*.pdf"))
    
    print(f"Found {len(pdf_files)} PDF files to process")
    
    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()
            
            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file'] = pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            
            all_documents.extend(documents)
            print(f"  ✓ Loaded {len(documents)} pages")
            
        except Exception as e:
            print(f"  ✗ Error: {e}")
    
    print(f"\nTotal documents loaded: {len(all_documents)}")
    return all_documents

# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")

Found 3 PDF files to process

Processing: pdf1.pdf


Ignoring wrong pointing object 6 0 (offset 0)
Ignoring wrong pointing object 8 0 (offset 0)
Ignoring wrong pointing object 10 0 (offset 0)
Ignoring wrong pointing object 12 0 (offset 0)
Ignoring wrong pointing object 14 0 (offset 0)
Ignoring wrong pointing object 16 0 (offset 0)
Ignoring wrong pointing object 18 0 (offset 0)
Ignoring wrong pointing object 20 0 (offset 0)
Ignoring wrong pointing object 22 0 (offset 0)
Ignoring wrong pointing object 24 0 (offset 0)
Ignoring wrong pointing object 26 0 (offset 0)
Ignoring wrong pointing object 28 0 (offset 0)
Ignoring wrong pointing object 30 0 (offset 0)
Ignoring wrong pointing object 32 0 (offset 0)
Ignoring wrong pointing object 34 0 (offset 0)
Ignoring wrong pointing object 40 0 (offset 0)
Ignoring wrong pointing object 45 0 (offset 0)
Ignoring wrong pointing object 53 0 (offset 0)


  ✓ Loaded 7 pages

Processing: pdf2.pdf
  ✓ Loaded 32 pages

Processing: pdf3.pdf
  ✓ Loaded 25 pages

Total documents loaded: 64


In [9]:
all_pdf_documents

[Document(metadata={'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20260329232124Z00'00'", 'title': 'data_centers6', 'author': 'gmarcy', 'moddate': "D:20260329232124Z00'00'", 'source': '..\\data\\pdf_files\\pdf1.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}, page_content='1\t\n      The Impact of Computing Data Centres Orbiting Earth   Geoffrey W. Marcy1*  1 Space Laser Awareness, 3388 Petaluma Hill Rd, Santa Rosa, CA, 95404, USA   Submitted to Letters of the Monthly Notices of the Royal Astronomical Society  Accepted: xx     Version: Mar 29, 2026   ABSTRACT  Artificial intelligence is projected to increase U.S. data-centre power demand beyond 100 gigawatt (GW) by 2035 and global demand toward 1 terrawatt. In response, companies and governments have proposed placing computing infrastructure in sun-synchronous low-Earth orbit, where continuous sunlight could supply electrical 

In [12]:
### Text splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n", " ", ""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")
    
    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")
    
    return split_docs


In [13]:
chunks=split_documents(all_pdf_documents)
chunks

Split 64 documents into 288 chunks

Example chunk:
Content: 1...
Metadata: {'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20260329232124Z00'00'", 'title': 'data_centers6', 'author': 'gmarcy', 'moddate': "D:20260329232124Z00'00'", 'source': '..\\data\\pdf_files\\pdf1.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20260329232124Z00'00'", 'title': 'data_centers6', 'author': 'gmarcy', 'moddate': "D:20260329232124Z00'00'", 'source': '..\\data\\pdf_files\\pdf1.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}, page_content='1'),
 Document(metadata={'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20260329232124Z00'00'", 'title': 'data_centers6', 'author': 'gmarcy', 'moddate': "D:20260329232124Z00'00'", 'source': '..\\data\\pdf_files\\pdf1.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}, page_content='The Impact of Computing Data Centres Orbiting Earth   Geoffrey W. Marcy1*  1 Space Laser Awareness, 3388 Petaluma Hill Rd, Santa Rosa, CA, 95404, USA   Submitted to Letters of the Monthly Notices of the Royal Astronomical Soc

In [14]:
class EmbeddingManager:
    """Class to handle embedding of documents using SentenceTransformer."""
    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initialize the embedding manager.
        Args:
            model_name (str): HuggingFace model name for sentence embedding.
        """
        self.model = model_name
        # self.model = None
        self._load_model()

    def _load_model(self):
        """Load the sentence transformer model."""
        try:
            print(f"Loading embedding model: {self.model}")
            self.model = SentenceTransformer(self.model)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of text strings.
        Args:
            texts (List[str]): Input texts to embed.
        Returns:
            np.ndarray: Generated embedding vectors.
        """
        if not self.model:
            raise ValueError("Model not loaded. Cannot generate embedding.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings of shape: {embeddings.shape}")
        return embeddings

## Initialize the embedding manager:
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7556.38it/s]


Model loaded successfully. Embedding dimension: 384


### Vector Store

In [15]:
class VectorStore:
    """Class to manage document embeddings in a ChromaDB vector store."""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """Initialize the vector store.
        Args:
            collection_name (str): Name of the ChromaDB collection to use.
            persist_directory (str): Directory to persist the vector store data.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_vector_store()

    def _initialize_vector_store(self):
        """Initialize the ChromaDB client and collection."""
        try:
            # Create persistent ChromaDB client
            print(f"Initializing ChromaDB client with persist directory: {self.persist_directory}")
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            print("ChromaDB client initialized successfully.")

            # Get or create the collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized with collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
        
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """Add documents to the vector store with their embeddings.
        Args:
            documents: List of Langchain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents and embeddings must match.")
        
        print(f"Adding {len(documents)} documents to the vector store...")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate a unique ID for each document and prepare metadata and text content
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"  # Generate a unique ID for each document
            ids.append(doc_id)
            # Prepare metadata
            metadata = dict(doc.metadata)  # Convert metadata to a dictionary if it's not already
            metadata['doc_index'] = i  # Add document index to metadata
            metadata['content_length'] = len(doc.page_content)  # Add content length to metadata
            metadatas.append(metadata)  # Use the prepared metadata
            # Document content
            documents_text.append(doc.page_content)  # Use the document's text content
            #Embeddings
            embeddings_list.append(embedding.tolist())  # Convert numpy array to list

        # Add data to the collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,  # Use the prepared embeddings list
                metadatas=metadatas,  # Use the prepared metadata list
                documents=documents_text  # Use the prepared document text list
            )
            print(f"Successfully added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore = VectorStore()
vectorstore


Initializing ChromaDB client with persist directory: ../data/vector_store
ChromaDB client initialized successfully.
Vector store initialized with collection: pdf_documents
Existing documents in collection: 0


In [16]:
chunks

[Document(metadata={'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20260329232124Z00'00'", 'title': 'data_centers6', 'author': 'gmarcy', 'moddate': "D:20260329232124Z00'00'", 'source': '..\\data\\pdf_files\\pdf1.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}, page_content='1'),
 Document(metadata={'producer': 'macOS Version 26.2 (Build 25C56) Quartz PDFContext', 'creator': 'Word', 'creationdate': "D:20260329232124Z00'00'", 'title': 'data_centers6', 'author': 'gmarcy', 'moddate': "D:20260329232124Z00'00'", 'source': '..\\data\\pdf_files\\pdf1.pdf', 'total_pages': 7, 'page': 0, 'page_label': '1', 'source_file': 'pdf1.pdf', 'file_type': 'pdf'}, page_content='The Impact of Computing Data Centres Orbiting Earth   Geoffrey W. Marcy1*  1 Space Laser Awareness, 3388 Petaluma Hill Rd, Santa Rosa, CA, 95404, USA   Submitted to Letters of the Monthly Notices of the Royal Astronomical Soc

In [17]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

Generating embeddings for 288 texts...


Batches: 100%|██████████| 9/9 [00:09<00:00,  1.01s/it]


Generated embeddings of shape: (288, 384)
Adding 288 documents to the vector store...
Successfully added 288 documents to the vector store.
Total documents in collection after addition: 288
